# Turkish Public Procurement Intelligence Project

Raw-to-dashboard analysis using Python, DuckDB SQL, Parquet, and Power BI.

Raw data coverage: 2010–2024.

In [1]:
import os
from pathlib import Path

import duckdb
import pandas as pd
from dotenv import load_dotenv

In [2]:
ROOT = Path.cwd()

if not (ROOT / ".env").exists() and (ROOT.parent / ".env").exists():
    ROOT = ROOT.parent

if not (ROOT / ".env").exists():
    raise FileNotFoundError(
        "Open the repository root in VS Code before running this file."
    )

load_dotenv(ROOT / ".env")

RAW_CSV = Path(os.environ["PROCUREMENT_RAW_CSV"])
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Raw file:", RAW_CSV)
print("Raw file exists:", RAW_CSV.exists())
print("Processed folder:", PROCESSED_DIR)

Project root: d:\All_projects\turkish-public-procurement-bi
Raw file: D:\All_projects\turkish-public-procurement-bi\data\raw\merged_contract_level_2010_2024.csv
Raw file exists: True
Processed folder: d:\All_projects\turkish-public-procurement-bi\data\processed


In [3]:
con = duckdb.connect()
print("duckdb connection opened ")

duckdb connection opened 


In [4]:
raw_csv_sql = RAW_CSV.as_posix().replace("'", " ' ' ")

con.execute(
    f"""
    CREATE OR REPLACE VIEW raw_contracts AS 
    SELECT *
    FROM read_csv_auto(
        '{raw_csv_sql}',
        all_varchar = true)
    """
)

print("Created DuckDB view: raw_contracts")

Created DuckDB view: raw_contracts


In [ ]:
# con.close()


In [14]:
row_count = con.execute(
    " SELECT COUNT(*) FROM raw_contracts "
).fetchone()[0]

print(f"Raw rows: {row_count:,}")

Raw rows: 2,370,736


In [16]:
sample = con.execute(
    """
    SELECT
        contract_id,
        tender_name,
        authority,
        province,
        supplier,
        contract_price
    FROM raw_contracts
    LIMIT 10
    """
).df()

sample

,contract_id,tender_name,authority,province,supplier,contract_price
0,26158,TAŞBAŞI KÖYÜ ÖĞRENCİLERİNİ TAŞIMA MERKEZİ ELCİ...,DALAMAN İLÇE MİLLİ EĞİTİM MÜDÜRLÜĞÜ,MUĞLA,NaN,0
1,143341,YAĞ,İSTANBUL ŞEHİR HATLARI TURİZM SANAYİ VE TİCARE...,İSTANBUL,PETROL OFİSİ ANONİM ŞİRKETİ,665882
2,48644,BAKIMSIZ AKÜ,BORU HATLARI İLE PETROL TAŞIMA A.Ş (BOTAŞ) Ted...,ANKARA,NaN,0
3,82381,TANDEM POMPA TAMİRİ (1 ADET),İSTON İSTANBUL BETON ELEMANL.VE HAZIR BET ON F...,İSTANBUL,HİDROER HİDROLİK PNÖ MATİK AKSAM SAN. VE TİC.L...,1500
4,541962,"Muhtelif marka, model ve tip iş makinesi ve at...",DEVLET MALZEME OFİSİ GENEL MÜDÜRLÜĞÜ(DMO) II N...,ANKARA,BORUSAN MAKİNA VE GÜ Ç SİSTEMLERİ SAN. VE TİC....,3856750
5,231698,BALİSTİKA 2010 Uygulama ve Veritabanı Sunucul...,TÜBİTAK UZAY TEKNOLOJİLERİ ARAŞTIRMA ENSTİTÜSÜ...,ANKARA,COMPRO BİLİŞİM TEKNOLOJİLERİ ANONİM ŞİRKETİ,471900
6,175211,FORD TRANSİT MİNİBÜS ALIMI,DEVLET MALZEME OFİSİ GENEL MÜDÜRLÜĞÜ(DMO) Gazi...,GAZİANTEP,FORD OTOMOTİV SANAYİ A.Ş.,35010
7,62965,"""""GAZ KOMPRESÖR YEDEKLERİ""""",BORU HATLARI İLE PETROL TAŞIMA A.Ş (BOTAŞ) Ted...,ANKARA,NaN,0
8,26338,DARIYERİ KÖYÜ ÖĞRENCİLERİNİ TAŞIMA MERKEZİ ÇÖĞ...,DALAMAN İLÇE MİLLİ EĞİTİM MÜDÜRLÜĞÜ,MUĞLA,NaN,0
9,25130,NARLI MAH. Ş.TURGUT YILMAZ İ.O. ÖĞRENCİLERİNİ ...,DALAMAN İLÇE MİLLİ EĞİTİM MÜDÜRLÜĞÜ,MUĞLA,NaN,0


In [17]:
# inspect the schema
schema = con.execute(
    "describe raw_contracts"
).df()
schema

,column_name,column_type,null,key,default,extra
0,tender_id,VARCHAR,YES,None,None,None
1,detail_tender_id,VARCHAR,YES,None,None,None
2,contract_id,VARCHAR,YES,None,None,None
3,announcement_id,VARCHAR,YES,None,None,None
4,ikn,VARCHAR,YES,None,None,None
5,tender_name,VARCHAR,YES,None,None,None
6,authority,VARCHAR,YES,None,None,None
7,authority_id,VARCHAR,YES,None,None,None
8,province,VARCHAR,YES,None,None,None
9,authority_district,VARCHAR,YES,None,None,None


In [18]:
# List column names
column_names = schema["column_name"].to_list()

print(f"column count: {len(column_names)}")
for name in column_names:
    print(name)

column count: 47
tender_id
detail_tender_id
contract_id
announcement_id
ikn
tender_name
authority
authority_id
province
authority_district
parent_authority
top_authority_code
top_authority_name
okas_code
okas_desc
okas_code_search
okas_codes_all
okas_names_all
okas_count
method
method_code
type
scope
is_electronic
is_partial
is_invitation_only
characteristics
document_count
status
status_code
tender_datetime
tender_date
tender_announcement_date
result_announcement_date
contract_date
days_announce_to_contract
total_estimated_cost
lot_estimated_cost
contract_price
supplier
num_offers
num_valid_offers
rebate
log_ratio
lot_estimate_missing
is_multi_lot
total_lots_in_tender


In [19]:
# View a small selection
con.execute(
    """
    SELECT
        contract_id,
        ikn,
        tender_name,
        authority,
        province,
        okas_code,
        method,
        type,
        contract_date,
        contract_price,
        supplier,
        num_valid_offers
    FROM raw_contracts
    LIMIT 10
    """
).df()

,contract_id,ikn,tender_name,authority,province,okas_code,method,type,contract_date,contract_price,supplier,num_valid_offers
0,26158,2010/500427,TAŞBAŞI KÖYÜ ÖĞRENCİLERİNİ TAŞIMA MERKEZİ ELCİ...,DALAMAN İLÇE MİLLİ EĞİTİM MÜDÜRLÜĞÜ,MUĞLA,80000000,İhale Usulü: Pazarlık (MD 21 F),Hizmet,None,0,NaN,0
1,143341,2010/582230,YAĞ,İSTANBUL ŞEHİR HATLARI TURİZM SANAYİ VE TİCARE...,İSTANBUL,NaN,İhale Usulü: 4734 / 3-g,Mal,None,665882,PETROL OFİSİ ANONİM ŞİRKETİ,7
2,48644,2010/573963,BAKIMSIZ AKÜ,BORU HATLARI İLE PETROL TAŞIMA A.Ş (BOTAŞ) Ted...,ANKARA,NaN,İhale Usulü: 4734 / 3-g,Mal,None,0,NaN,0
3,82381,2011/9114,TANDEM POMPA TAMİRİ (1 ADET),İSTON İSTANBUL BETON ELEMANL.VE HAZIR BET ON F...,İSTANBUL,NaN,İhale Usulü: 4734 / 3-g,Hizmet,None,1500,HİDROER HİDROLİK PNÖ MATİK AKSAM SAN. VE TİC.L...,2
4,541962,2012/77413,"Muhtelif marka, model ve tip iş makinesi ve at...",DEVLET MALZEME OFİSİ GENEL MÜDÜRLÜĞÜ(DMO) II N...,ANKARA,NaN,İhale Usulü: 4734 / 3-g,Mal,None,3856750,BORUSAN MAKİNA VE GÜ Ç SİSTEMLERİ SAN. VE TİC....,1
5,231698,2011/87707,BALİSTİKA 2010 Uygulama ve Veritabanı Sunucul...,TÜBİTAK UZAY TEKNOLOJİLERİ ARAŞTIRMA ENSTİTÜSÜ...,ANKARA,NaN,İhale Usulü: 4734 / 3-f,Mal,None,471900,COMPRO BİLİŞİM TEKNOLOJİLERİ ANONİM ŞİRKETİ,3
6,175211,2011/77301,FORD TRANSİT MİNİBÜS ALIMI,DEVLET MALZEME OFİSİ GENEL MÜDÜRLÜĞÜ(DMO) Gazi...,GAZİANTEP,NaN,İhale Usulü: 4734 / 3-g,Mal,None,35010,FORD OTOMOTİV SANAYİ A.Ş.,1
7,62965,2010/579814,"""""GAZ KOMPRESÖR YEDEKLERİ""""",BORU HATLARI İLE PETROL TAŞIMA A.Ş (BOTAŞ) Ted...,ANKARA,NaN,İhale Usulü: 4734 / 3-g,Mal,None,0,NaN,0
8,26338,2010/500462,DARIYERİ KÖYÜ ÖĞRENCİLERİNİ TAŞIMA MERKEZİ ÇÖĞ...,DALAMAN İLÇE MİLLİ EĞİTİM MÜDÜRLÜĞÜ,MUĞLA,80000000,İhale Usulü: Pazarlık (MD 21 F),Hizmet,None,0,NaN,0
9,25130,2010/500357,NARLI MAH. Ş.TURGUT YILMAZ İ.O. ÖĞRENCİLERİNİ ...,DALAMAN İLÇE MİLLİ EĞİTİM MÜDÜRLÜĞÜ,MUĞLA,80000000,İhale Usulü: Pazarlık (MD 21 F),Hizmet,None,0,NaN,0


In [20]:
# %%
missing_critical = con.execute(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(contract_id) AS missing_contract_id,
        COUNT(*) - COUNT(authority_id) AS missing_authority_id,
        COUNT(*) - COUNT(supplier) AS missing_supplier,
        COUNT(*) - COUNT(contract_date) AS missing_contract_date,
        COUNT(*) - COUNT(contract_price) AS missing_contract_price,
        COUNT(*) - COUNT(okas_code) AS missing_okas_code,
        COUNT(*) - COUNT(num_valid_offers) AS missing_valid_offers
    FROM raw_contracts
    """
).df()

missing_critical

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,missing_contract_id,missing_authority_id,missing_supplier,missing_contract_date,missing_contract_price,missing_okas_code,missing_valid_offers
0,2370736,0,12911,75139,911382,0,399818,0


In [21]:
# Important: empty strings may not be SQL NULL. Check both:
# %%
blank_critical = con.execute(
    """
    SELECT
        SUM(CASE WHEN TRIM(COALESCE(contract_id, '')) = '' THEN 1 ELSE 0 END)
            AS blank_contract_id,
        SUM(CASE WHEN TRIM(COALESCE(supplier, '')) = '' THEN 1 ELSE 0 END)
            AS blank_supplier,
        SUM(CASE WHEN TRIM(COALESCE(contract_price, '')) = '' THEN 1 ELSE 0 END)
            AS blank_contract_price
    FROM raw_contracts
    """
).df()

blank_critical

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,blank_contract_id,blank_supplier,blank_contract_price
0,0.0,75139.0,0.0


In [22]:
# Distinct counts
# %%
distinct_counts = con.execute(
    """
    SELECT
        COUNT(DISTINCT tender_id) AS tenders,
        COUNT(DISTINCT contract_id) AS contracts,
        COUNT(DISTINCT authority_id) AS authorities,
        COUNT(DISTINCT supplier) AS supplier_names,
        COUNT(DISTINCT province) AS provinces,
        COUNT(DISTINCT okas_code) AS okas_codes,
        COUNT(DISTINCT method_code) AS method_codes,
        COUNT(DISTINCT type) AS procurement_types
    FROM raw_contracts
    """
).df()

distinct_counts

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,tenders,contracts,authorities,supplier_names,provinces,okas_codes,method_codes,procurement_types
0,1557241,2370077,29259,230369,82,6929,26,3


In [23]:
# Category values
# %%
con.execute(
    """
    SELECT
        type,
        COUNT(*) AS contracts
    FROM raw_contracts
    GROUP BY type
    ORDER BY contracts DESC
    """
).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,type,contracts
0,Mal,1411022
1,Hizmet,657439
2,Yapım,289364
3,NaN,12911


In [25]:
# method values
con.execute(
    """
    SELECT
        method,
        COUNT(*) AS contracts
    FROM raw_contracts
    GROUP BY method
    ORDER BY contracts DESC
    """
).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,method,contracts
0,İhale Usulü: Açık,1563288
1,İhale Usulü: Pazarlık (MD 21 F),241557
2,İhale Usulü: Pazarlık (MD 21 B),185211
3,İhale Usulü: 4734 / 3-g,161692
4,İhale Usulü: Diğer,126266
5,İhale Usulü: Pazarlık (Hepsi),27833
6,İhale Usulü: Belli İstekliler Arasında,24183
7,NaN,12911
8,İhale Usulü: 4734 / 3-a,8846
9,İhale Usulü: Pazarlık (MD 21 C),5673


In [26]:
# method values
con.execute(
    """
    SELECT
        method_code,
        COUNT(*) AS contracts
    FROM raw_contracts
    GROUP BY method_code
    ORDER BY contracts DESC
    """
).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,method_code,contracts
0,1,1563288
1,3,461591
2,16,161692
3,50,126266
4,2,24183
5,NaN,15589
6,10,8846
7,15,3354
8,18,2112
9,14,1985


In [27]:
# scope
con.execute(
    """
    SELECT
        scope,
        COUNT(*) AS contracts
    FROM raw_contracts
    GROUP BY scope
    ORDER BY  contracts DESC
    """
).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,scope,contracts
0,4734 Kapsamında,1964575
1,İstisna,390415
2,NaN,12911
3,Kapsam Dışı,2835


In [28]:
# status
con.execute(
    """
    select
        status,
        count(*) as contracts
    from raw_contracts
    group by status
    order by contracts desc
    """
).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,status,contracts
0,Sonuç İlanı Yayımlanmış,2357825
1,NaN,12911


In [29]:
# province
con.execute(
    """
    select
        province,
        count(*) as contracts
    from raw_contracts
    group by province
    order by contracts desc
    """
).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,province,contracts
0,İSTANBUL,258171
1,ANKARA,242889
2,İZMİR,155730
3,KOCAELİ,63302
4,BURSA,50803
...,...,...
77,BARTIN,7305
78,ARDAHAN,7212
79,IĞDIR,6889
80,KİLİS,6261


In [30]:
con.execute(
    """
    select
        is_multi_lot,
        count(*) as contracts
    from raw_contracts
    group by is_multi_lot
    order by contracts desc
    """
).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,is_multi_lot,contracts
0,0,1371194
1,1,999542
